# XGBoost v2 - trening i analiza predykcji

Notebook korzysta z aktualnej struktury pakietu `accur8pool`:

* `accur8pool.files_utils.filesManager.FilesManager` - parowanie plikow danych z etykietami,
* `accur8pool.ML.dataset` - skladanie zbioru (`build_training_frame`) i podzial po plikach (`split_by_file`),
* `accur8pool.ML.pipelines` - `PipelineInit`, `make_pipeline`, `XGBPipeline` (okienkowanie, cechy, model, wygladzanie),
* `accur8pool.scripts.ploting.to_segments` - segmenty etykiet do wykresow.

Oczekiwana struktura katalogow (lustrzana - plik danych i etykiet maja te sama sciezke wzgledna):

```
data/
  prepared/<podkatalog>/<nazwa>.csv   # wynik `accur8pool prepare`
  labeled/<podkatalog>/<nazwa>.csv    # etykiety z kolumna `label`
```

Pliki z `prepared/` bez odpowiednika w `labeled/` sa pomijane (lista ponizej).

In [ ]:
import os
from pathlib import Path

# notebook lezy w notebooks/ - sciezki ponizej sa wzgledem katalogu glownego repo
if Path.cwd().name == 'notebooks':
    os.chdir('..')
Path.cwd()

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import classification_report

from accur8pool.data_processing.const import DATA_COLUMNS_TO_TRAIN, LABEL
from accur8pool.files_utils.filesManager import FilesManager
from accur8pool.ML.dataset import (
    FILE_ID,
    build_training_frame,
    split_by_file,
    split_by_index,
    split_features_labels,
)
from accur8pool.ML.pipelines import PipelineInit, XGBPipeline, make_pipeline
from accur8pool.scripts.ploting import to_segments

## Konfiguracja

In [ ]:
DATA_ROOT = Path('data')
DATA_SUBDIR = 'prepared'
LABELS_SUBDIR = 'labeled'

FEATURE_COLUMNS = list(DATA_COLUMNS_TO_TRAIN)
LABEL_COLUMN = LABEL

TRAIN_FRAC, TEST_FRAC = 0.7, 0.2   # reszta (0.1) -> eval, uzywany do early stopping

WINDOW_SIZE = 20
WINDOW_STEP = 3
MIN_GROUP_SIZE = 7                 # wygladzanie krotkich segmentow predykcji

MODEL_PARAMS = {'max_depth': 6, 'subsample': 0.8, 'learning_rate': 0.1}
N_ESTIMATORS = 2000
EARLY_STOPPING_ROUNDS = 10

MODEL_PATH = Path('models') / 'xgboostv2.pkl'
SAVE_MODEL = False

## Wczytanie danych

In [ ]:
files_manager = FilesManager(root_dir=DATA_ROOT, data_subdir=DATA_SUBDIR, labels_subdir=LABELS_SUBDIR)
data_label_paths = files_manager.get_data_label_paths()
unlabeled = files_manager.get_unlabeled_data()

print(f'par dane-etykiety: {len(data_label_paths)}, plikow bez etykiet: {len(unlabeled)}')
data_label_paths[:5]

In [ ]:
frame = build_training_frame(files_manager, label_column=LABEL_COLUMN)
frame.shape, frame[FILE_ID].nunique()

## Podzial train / test / eval

Domyslnie po plikach (`split_by_file`) - caly plik trafia do jednego zbioru, wiec okna z sasiednich
probek nie przeciekaja miedzy train a test. Przy mniej niz 3 plikach zostaje podzial po indeksie.

In [ ]:
splitter = split_by_file if frame[FILE_ID].nunique() >= 3 else split_by_index
train_df, test_df, eval_df = splitter(frame, TRAIN_FRAC, TEST_FRAC)

X_train, y_train = split_features_labels(train_df, FEATURE_COLUMNS, LABEL_COLUMN)
X_test, y_test = split_features_labels(test_df, FEATURE_COLUMNS, LABEL_COLUMN)
X_eval, y_eval = split_features_labels(eval_df, FEATURE_COLUMNS, LABEL_COLUMN)

print(splitter.__name__)
X_train.shape, X_test.shape, X_eval.shape

In [ ]:
pd.DataFrame({
    'train': y_train.value_counts(normalize=True),
    'test': y_test.value_counts(normalize=True),
    'eval': y_eval.value_counts(normalize=True),
}).sort_index()

## Trening

`fit_prepared` okienkuje dane (`DataWindowing` + `CenterLabelingStrategy`), liczy cechy okien
(`extractWindowFeatures`) i wagi probek (`sqrt_balanced_weights`), a `eval` sluzy do early stopping.

In [ ]:
num_class = int(y_train.max()) + 1
assert sorted(y_train.unique()) == list(range(num_class)), 'XGBoost wymaga klas 0..num_class-1 w zbiorze treningowym'

pipeline = make_pipeline(PipelineInit(
    model_params=MODEL_PARAMS,
    window_size=WINDOW_SIZE,
    window_step=WINDOW_STEP,
    model_n_estimators=N_ESTIMATORS,
    model_early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    num_class=num_class,
))
pipeline.DATA_COLUMNS = FEATURE_COLUMNS

pipeline.fit_prepared(X_train, y_train, X_eval, y_eval, summary_path=None, verbose=50)
pipeline.model.best_iteration

## Ewaluacja na zbiorze testowym (poziom okien)

In [ ]:
metrics = pipeline.evaluate_prepared(X_test, y_test)
print(f"okien: {metrics['n_windows']}")
print(f"f1_weighted       = {metrics['f1_weighted']:.4f}")
print(f"balanced_accuracy = {metrics['balanced_accuracy']:.4f}")
pd.DataFrame(metrics['per_class']).T

In [ ]:
windowing = pipeline.windowing

X_test_windowed, y_test_windowed = windowing.window_with_extractions(
    X_test[FEATURE_COLUMNS].to_numpy(), y_test.to_numpy()
)
y_pred = pipeline.model.predict(X_test_windowed)
smoothed = XGBPipeline.smooth(y_pred, MIN_GROUP_SIZE)

print(classification_report(y_test_windowed, smoothed, zero_division=0))

## Etykiety referencyjne vs predykcja vs predykcja wygladzona

In [ ]:
st, sp = 0, min(len(y_pred), 500)   # zakres okien do podgladu

compare = pd.DataFrame({
    'reference': y_test_windowed[st:sp],
    'predicted': y_pred[st:sp],
    'smoothed': smoothed[st:sp],
}, index=np.arange(st, sp))

fig = px.scatter(compare.melt(ignore_index=False, var_name='serie', value_name='label').reset_index(),
                 x='index', y='label', color='serie', facet_row='serie', height=600)
fig

## Wykrywanie pelnych sekwencji uderzenia

Szukamy kolejnych segmentow wygladzonej predykcji ukladajacych sie w sekwencje faz `1 -> 2 -> 3 -> 4 -> 0`.

In [ ]:
SHOT_SEQUENCE = [1, 2, 3, 4, 0]


def find_sequences(labels, sequence=SHOT_SEQUENCE):
    """Zwraca (start, end) w indeksach okien dla kazdego wystapienia sekwencji segmentow."""
    segments = to_segments(labels)
    seg_labels = [int(label) for _, _, label in segments]
    n = len(sequence)

    found = []
    for i in range(len(segments) - n + 1):
        if seg_labels[i:i + n] == sequence:
            found.append((int(segments[i][0]), int(segments[i + n - 1][1])))

    return found


pred_shots = find_sequences(smoothed)
ref_shots = find_sequences(y_test_windowed)
print(f'sekwencji w predykcji: {len(pred_shots)}, w etykietach: {len(ref_shots)}')
pred_shots[:10]

In [ ]:
if pred_shots:
    start, end = pred_shots[0]
    px.scatter(x=np.arange(start, end), y=smoothed[start:end], color=smoothed[start:end].astype(str),
               labels={'x': 'okno', 'y': 'label', 'color': 'label'}).show()

## Predykcja dla pojedynczego pliku

`predict_prepared` okienkuje plik, przewiduje, wygladza i rozwija etykiety okien z powrotem na probki
(`reverse_window_labels`), wiec wynik ma dlugosc pliku. Bierzemy pierwszy plik ze zbioru testowego.

In [ ]:
test_file_id = int(test_df[FILE_ID].iloc[0])
data_path, label_path = data_label_paths[test_file_id]

file_df = frame[frame[FILE_ID] == test_file_id].reset_index(drop=True)
file_pred = pipeline.predict_prepared(file_df, min_group_size=MIN_GROUP_SIZE)

print(data_path)
print(classification_report(file_df[LABEL_COLUMN].astype(int), file_pred.astype(int), zero_division=0))

In [ ]:
LABEL_COLORS = {
    0: 'rgba(0,0,0,0.20)',
    1: 'rgba(102,255,51,0.20)',
    2: 'rgba(0,153,255,0.20)',
    3: 'rgba(255,0,0,0.20)',
    4: 'rgba(255,255,153,0.20)',
}


def plot_signal_with_labels(signal: pd.DataFrame, labels, columns, title=None):
    """Sygnal (min-max) z nalozonymi segmentami etykiet."""
    scaled = (signal[columns] - signal[columns].min()) / (signal[columns].max() - signal[columns].min() + 1e-9)
    x = np.arange(len(scaled))

    fig = go.Figure()
    for col in columns:
        fig.add_trace(go.Scatter(x=x, y=scaled[col], mode='lines', name=col))

    for start, end, label in to_segments(labels):
        fig.add_vrect(x0=start, x1=end, fillcolor=LABEL_COLORS.get(int(label), 'rgba(0,0,0,0.1)'),
                      opacity=0.7, line_width=0, layer='below')

    fig.update_layout(title=title)
    return fig


PLOT_COLUMNS = ['roll', 'acc_magnitude_jerk']
plot_signal_with_labels(file_df, file_pred, PLOT_COLUMNS, title=f'predykcja: {data_path.name}')

In [ ]:
plot_signal_with_labels(file_df, file_df[LABEL_COLUMN], PLOT_COLUMNS, title=f'etykiety: {label_path.name}')

## Zapis modelu

Ten sam format co `accur8pool train` - mozna go uzyc w `accur8pool predict -m ...`.

In [ ]:
if SAVE_MODEL:
    MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
    pipeline.save_model(MODEL_PATH)
    pipeline._make_fit_summary(MODEL_PATH.with_suffix('.json'))
    print(f'zapisano {MODEL_PATH}')